# Synthetic Data Generation & Evaluation — Multi-Synthesizer Benchmark

Pipeline over three tables (`CONTACT`, `PERSON`, `PERSONNAME`), comparing **several SDV synthesizers**:

| Synthesizer | Type | Notes |
|---|---|---|
| `HMA` | multi-table | your original HMASynthesizer |
| `GaussianCopula` | single-table | fast statistical model |
| `CTGAN` | single-table | GAN-based (needs torch, slower) |
| `TVAE` | single-table | variational autoencoder |
| `CopulaGAN` | single-table | copula-normalised CTGAN |

Single-table models are valid here because relationships are removed → tables are independent.

**Flow:** read CSVs → holdout split (before fit!) → metadata → fit + sample every synthesizer →
**Fidelity** (QualityReport: Column Shapes + Column Pair Trends, per-column heatmaps) →
**Visualization** (distribution overlays, correlation heatmaps, HTML report) →
**Privacy** (membership inference attack, DCR overlays, exact-match, sdmetrics) →
**Utility** (TSTR comparison) → **leaderboard heatmap** + `reports/summary.json`.

All heavy logic lives in `synth_eval.py`. Everything auto-detects numeric vs categorical
columns, handles missing values, and skips id/name columns.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import synth_eval as se  # local helper module (synth_eval.py)

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------ config
SEED_DIR     = 'sdg/seed'
REPORTS_DIR  = 'reports'
RANDOM_STATE = 42
HOLDOUT_FRAC = 0.25      # portion of each real table reserved BEFORE fitting
SAMPLE_SCALE = 1.0       # synthetic rows = scale * training rows

# Which synthesizers to benchmark. Remove entries to speed things up.
SYNTHESIZERS = ['HMA', 'GaussianCopula', 'CTGAN', 'TVAE', 'CopulaGAN']
PRIMARY      = 'HMA'     # gets the full per-column detailed report
GAN_EPOCHS   = 300       # CTGAN/TVAE/CopulaGAN epochs — drop to 50 for a quick pass

# Optional per-table ML target override; leave a table out to auto-select.
# e.g. TARGETS = {'PERSON': 'gender_code'}
TARGETS = {}

os.makedirs(os.path.join(REPORTS_DIR, 'figures'), exist_ok=True)
np.random.seed(RANDOM_STATE)
import sdv, sdmetrics
print('sdv =', sdv.__version__, '| sdmetrics =', sdmetrics.__version__)

## 1. Read the data

In [ ]:
df_con = pd.read_csv(os.path.join(SEED_DIR, 'CONTACT.csv'))
df_per = pd.read_csv(os.path.join(SEED_DIR, 'PERSON.csv'))
df_pnm = pd.read_csv(os.path.join(SEED_DIR, 'PERSONNAME.csv'))

real_tables = {'CONTACT': df_con, 'PERSON': df_per, 'PERSONNAME': df_pnm}
for name, df in real_tables.items():
    print(f'{name:12s} shape={df.shape}')
df_con.head()

## 2. Holdout split **before** fitting
Holdout rows are never seen by any synthesizer — they are the *non-members* for the
membership-inference attack and the *test set* for TSTR.

In [ ]:
from sklearn.model_selection import train_test_split

train_tables, holdout_tables = {}, {}
for name, df in real_tables.items():
    if len(df) < 8:
        train_tables[name], holdout_tables[name] = df.copy(), df.iloc[0:0].copy()
        continue
    tr, ho = train_test_split(df, test_size=HOLDOUT_FRAC, random_state=RANDOM_STATE, shuffle=True)
    train_tables[name]   = tr.reset_index(drop=True)
    holdout_tables[name] = ho.reset_index(drop=True)
    print(f'{name:12s} train={len(tr):6d}  holdout={len(ho):6d}')

## 3. Metadata (relationships removed → independent tables)

In [ ]:
from sdv.metadata import Metadata

metadata = Metadata.detect_from_dataframes(train_tables)

meta_dict = metadata.to_dict()
if meta_dict.get('relationships'):
    print('Removing detected relationships:', meta_dict['relationships'])
    meta_dict['relationships'] = []
    metadata = Metadata.load_from_dict(meta_dict)

meta_dicts = metadata.to_dict()['tables']   # per-table dicts for sdmetrics
metadata.validate()

# Column roles (numeric / categorical / skipped id-name) reused everywhere.
table_roles = {name: se.classify_columns(train_tables[name], metadata, name)
               for name in real_tables}
for name, r in table_roles.items():
    print(f'{name:12s} numeric={r.numeric}\n{"":12s} categorical={r.categorical}\n{"":12s} skipped={r.skipped}')

## 4. Fit & sample **every** synthesizer
`HMA` = multi-table (original flow); the rest are single-table models fitted per table.
A synthesizer that fails (e.g. torch not installed) is skipped with a warning.

In [ ]:
synth_suite = se.generate_synthetic_suite(
    train_tables, metadata,
    synthesizers=SYNTHESIZERS, scale=SAMPLE_SCALE, epochs=GAN_EPOCHS,
)
if PRIMARY not in synth_suite:
    PRIMARY = list(synth_suite)[0]
    print(f'Primary synthesizer unavailable, falling back to {PRIMARY}')

synth_data = synth_suite[PRIMARY]   # keeps the original single-synthesizer name alive
print('\nBenchmarking synthesizers:', list(synth_suite))

## 5. Fidelity — QualityReport for every synthesizer
Column Shapes + Column Pair Trends per synthesizer per table, then:
* grouped-bar comparison of all scores,
* per-column shape-score heatmap (columns × synthesizers) per table.

In [ ]:
from sdmetrics.reports.single_table import QualityReport

quality_scores = {}          # {synth: {table: {overall, shapes, pairs}}}
shape_details  = {}          # {table: {synth: Series(column -> score)}}

for sname, tables in synth_suite.items():
    quality_scores[sname] = {}
    for tname in real_tables:
        qr = QualityReport()
        qr.generate(train_tables[tname], tables[tname], meta_dicts[tname], verbose=False)
        props = qr.get_properties().set_index('Property')['Score'].to_dict()
        quality_scores[sname][tname] = {
            'overall': float(qr.get_score()),
            'column_shapes': float(props.get('Column Shapes', float('nan'))),
            'column_pair_trends': float(props.get('Column Pair Trends', float('nan'))),
        }
        det = qr.get_details('Column Shapes')
        shape_details.setdefault(tname, {})[sname] = det.set_index('Column')['Score']
        q = quality_scores[sname][tname]
        print(f"{sname:15s} {tname:12s} overall={q['overall']:.4f}  "
              f"shapes={q['column_shapes']:.4f}  pairs={q['column_pair_trends']:.4f}")

In [ ]:
from IPython.display import Image, display

comparison_figs = []   # (title, b64) — collected for the HTML report

b64 = se.plot_quality_comparison(quality_scores, os.path.join(REPORTS_DIR, 'figures', 'quality_comparison.png'))
comparison_figs.append(('Fidelity: QualityReport scores by synthesizer', b64))
display(Image(os.path.join(REPORTS_DIR, 'figures', 'quality_comparison.png')))

for tname, per_synth in shape_details.items():
    png = os.path.join(REPORTS_DIR, 'figures', f'column_shapes_{tname}.png')
    b64 = se.plot_column_shapes_heatmap(per_synth, png, tname)
    if b64:
        comparison_figs.append((f'{tname}: per-column shape scores', b64))
        display(Image(png))

## 6. Visualization
* Full per-column real-vs-synthetic report for the **primary** synthesizer (PNG + HTML)
* **Overlay plots**: real vs *all* synthesizers on the same axes for the top columns
* Correlation heatmaps real vs synthetic

In [ ]:
viz_artifacts = []
for name in real_tables:
    art = se.visualize_table(train_tables[name], synth_data[name], table_roles[name],
                             name, REPORTS_DIR, metadata=metadata)
    viz_artifacts.append(art)
    print(f'{name}: {len(art["columns"])} column plots'
          + (' + correlation heatmap' if art['correlation_b64'] else ''))

# Multi-synthesizer overlays (top columns of each table)
for name in real_tables:
    synths_for_table = {s: tbls[name] for s, tbls in synth_suite.items()}
    overlays = se.visualize_table_multi(train_tables[name], synths_for_table,
                                        table_roles[name], name, REPORTS_DIR, max_cols=8)
    comparison_figs.extend(overlays)
    print(f'{name}: {len(overlays)} overlay plots')

# Show one example overlay inline
if comparison_figs:
    import glob
    ex = sorted(glob.glob(os.path.join(REPORTS_DIR, 'figures', '*', 'multi_dist_*.png')))
    if ex:
        display(Image(ex[0]))

## 7. Privacy — every synthesizer
Membership inference attack (train members vs unseen holdout), DCR vs real→real baseline,
exact-match rate, sdmetrics `NewRowSynthesis`/`CategoricalCAP` — with pass/warn verdicts,
a 3-panel comparison dashboard and per-table DCR overlays.

In [ ]:
privacy_all = {}    # {synth: {table: report}}
for sname, tables in synth_suite.items():
    privacy_all[sname] = {}
    for tname in real_tables:
        rep = se.privacy_report(train_tables[tname], holdout_tables[tname], tables[tname],
                                table_roles[tname], tname, REPORTS_DIR, metadata=metadata)
        privacy_all[sname][tname] = rep
        mia = rep['membership_inference']
        auc = mia.get('auc')
        auc_s = f"{auc:.3f}" if auc == auc else 'n/a'
        print(f'--- {sname} / {tname}:  MIA AUC={auc_s}  '
              + '  '.join(f"{k}={v['status']}" for k, v in rep['verdicts'].items()))

In [ ]:
# 3-panel privacy dashboard (MIA AUC / DCR ratio / exact-match %)
png = os.path.join(REPORTS_DIR, 'figures', 'privacy_comparison.png')
b64 = se.plot_privacy_comparison(privacy_all, png)
if b64:
    comparison_figs.append(('Privacy metrics by synthesizer', b64))
    display(Image(png))

# DCR overlay per table: every synthesizer's real->synth distances vs baseline
for tname in real_tables:
    arrays, baseline = {}, None
    for sname in privacy_all:
        arr = privacy_all[sname][tname].get('dcr_arrays', {})
        if arr.get('real_synth') is not None:
            arrays[sname] = arr['real_synth']
            baseline = arr['real_real'] if baseline is None else baseline
    png = os.path.join(REPORTS_DIR, 'figures', tname, 'dcr_overlay.png')
    b64 = se.plot_dcr_overlay(arrays, baseline, png, tname)
    if b64:
        comparison_figs.append((f'{tname}: DCR overlay (all synthesizers)', b64))
        display(Image(png))

## 8. ML Efficacy — TSTR across all synthesizers
Same RandomForest + logistic/linear baseline trained on real **and on each synthesizer's data**,
all tested on the identical real holdout. Saved to `reports/ml_efficacy_tstr.csv` with
`gap(real-<synth>)` rows, plus a grouped-bar comparison figure.

In [ ]:
efficacy_frames = []
for name in real_tables:
    roles = table_roles[name]
    if name in TARGETS and TARGETS[name] in train_tables[name].columns:
        target = TARGETS[name]
        nun = train_tables[name][target].nunique(dropna=True)
        task = 'classification' if (target in roles.categorical or 2 <= nun <= 20) else 'regression'
    else:
        pick = se.auto_select_target(train_tables[name], roles)
        if pick is None:
            print(f'{name}: no suitable target, skipping TSTR')
            continue
        target, task = pick
    print(f'{name}: target="{target}" task={task}')
    synths_for_table = {s: tbls[name] for s, tbls in synth_suite.items()}
    frame = se.ml_efficacy_tstr(train_tables[name], holdout_tables[name], synths_for_table,
                                roles, target, task, table_name=name, random_state=RANDOM_STATE)
    efficacy_frames.append(frame)

efficacy_table = pd.concat(efficacy_frames, ignore_index=True) if efficacy_frames else pd.DataFrame()
if not efficacy_table.empty:
    csv_path = os.path.join(REPORTS_DIR, 'ml_efficacy_tstr.csv')
    efficacy_table.to_csv(csv_path, index=False)
    print('Saved', csv_path)
display(efficacy_table)

In [ ]:
png = os.path.join(REPORTS_DIR, 'figures', 'efficacy_comparison.png')
b64 = se.plot_efficacy_comparison(efficacy_table, png)
if b64:
    comparison_figs.append(('ML efficacy (TSTR) by training source', b64))
    display(Image(png))

## 9. Leaderboard + consolidated summary
0–1 scores per synthesizer on **fidelity / privacy / utility**, rendered as an annotated
heatmap, saved with everything else into `reports/summary.json`, `reports/leaderboard.csv`
and the single HTML report `reports/report.html`.

In [ ]:
leaderboard = se.compute_leaderboard(quality_scores, privacy_all, efficacy_table)
leaderboard.to_csv(os.path.join(REPORTS_DIR, 'leaderboard.csv'))
png = os.path.join(REPORTS_DIR, 'figures', 'leaderboard.png')
b64 = se.plot_leaderboard(leaderboard, png)
if b64:
    comparison_figs.insert(0, ('Synthesizer leaderboard', b64))
    display(Image(png))
display(leaderboard)

In [ ]:
def _clean(o):
    """Make numpy / nan values JSON-serialisable; drop raw arrays."""
    if isinstance(o, dict):
        return {k: _clean(v) for k, v in o.items() if k != 'dcr_arrays'}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return None
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating, float)):
        return None if (isinstance(o, float) and np.isnan(o)) else float(o)
    return o

summary = _clean({
    'config': {'holdout_frac': HOLDOUT_FRAC, 'sample_scale': SAMPLE_SCALE,
               'random_state': RANDOM_STATE, 'synthesizers': list(synth_suite),
               'primary': PRIMARY, 'gan_epochs': GAN_EPOCHS},
    'leaderboard': leaderboard.round(4).to_dict('index'),
    'fidelity_quality': quality_scores,
    'privacy': {s: {t: {'verdicts': privacy_all[s][t]['verdicts'],
                        'mia': privacy_all[s][t]['membership_inference'],
                        'dcr': privacy_all[s][t]['dcr'],
                        'exact_match': privacy_all[s][t]['exact_match'],
                        'sdmetrics': privacy_all[s][t]['sdmetrics']}
                    for t in privacy_all[s]}
                for s in privacy_all},
    'ml_efficacy_gaps': (efficacy_table[efficacy_table['train_on'].astype(str).str.startswith('gap(')]
                         .to_dict('records') if not efficacy_table.empty else []),
})

with open(os.path.join(REPORTS_DIR, 'summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=2)

html_path = se.build_html_report(viz_artifacts, os.path.join(REPORTS_DIR, 'report.html'),
                                 extra_summary={'leaderboard': summary['leaderboard'],
                                                'fidelity_quality': summary['fidelity_quality']},
                                 comparison_figures=comparison_figs)

print('=' * 70)
print('CONSOLIDATED SUMMARY')
print('=' * 70)
print('\nLeaderboard (1.0 = best):')
print(leaderboard.round(3).to_string())
for sname in synth_suite:
    print(f'\n[{sname}]')
    for tname in real_tables:
        q = quality_scores[sname][tname]
        pv = privacy_all[sname][tname]['verdicts']
        print(f"  {tname:12s} fidelity={q['overall']:.3f}  "
              + '  '.join(f"{k}={v['status']}" for k, v in pv.items()))
print('\nSaved: reports/summary.json | reports/leaderboard.csv |', html_path)